# Module 08 — Classes and Encapsulation

## Exercise 08.3 — Six leaks

Each class below hands out something it should not, or accepts something it
should have copied. Find the leak, fix it, and write the test that proves it.
The question to ask of every method: "after this returns, who else can reach
this object, and what can they do to it?"
Run:  python ex03_encapsulation.py

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.

---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 2. The attribute-lookup ladder

**The single most useful diagram in Part 2.** When you write `obj.x`, Python:

```text
1. type(obj).__mro__  -- looking for a DATA DESCRIPTOR named x
                         (something with __get__ AND __set__ -- e.g. @property)
                         found? call its __get__ and STOP.
2. obj.__dict__['x']  -- the instance's own dictionary
                         found? return it and STOP.
3. type(obj).__mro__  -- the class and its bases, in MRO order
                         found? return it (binding it if it is a function)
4. type(obj).__getattr__('x')   -- last-resort hook, if defined
5. AttributeError
```


Two consequences that explain a great deal:

**Instance attributes shadow class attributes** (step 2 beats step 3) — but
**properties beat instance attributes** (step 1 beats step 2). That ordering is
what makes `@property` able to intercept an attribute that used to be plain
data.

**A method is found on the class, not the instance.** Every instance of a class
shares one function object; the binding happens at lookup time.

In [ ]:
class Dog:
    def speak(self): return "woof"

d = Dog()
Dog.speak            # <function Dog.speak>       -- a plain function
d.speak              # <bound method Dog.speak>   -- function + instance
d.speak()            # == Dog.speak(d)

That is all `self` is: the first parameter, filled in by the binding. Python
makes it explicit rather than implicit, which is why you can do this:

In [ ]:
Dog.speak(d)                      # call it unbound
handler = d.speak                 # store a bound method as a callback
list(map(str.upper, ["a", "b"]))  # use an unbound method as a function

---

## Concept 3. Class attributes versus instance attributes

In [ ]:
class Counter:
    count = 0                     # CLASS attribute -- one, shared

    def __init__(self):
        self.items = []           # INSTANCE attribute -- one per object

The trap, and it is Module 02 wearing a class costume:

In [ ]:
class Basket:
    contents = []                 # SHARED between every instance

a, b = Basket(), Basket()
a.contents.append("apple")        # MUTATES the shared list
print(b.contents)                 # ['apple']   <-- !

Whereas:

In [ ]:
a.contents = ["apple"]            # REBINDS: creates an INSTANCE attribute
print(b.contents)                 # []          -- b still sees the class one

Mutation hits the shared object; assignment creates a per-instance shadow. Same
two operations from Module 02, same opposite outcomes.

**Rule: mutable state goes in `__init__`.** Class attributes are for constants,
defaults that are immutable, and things genuinely shared by all instances (a
registry, a counter of instances created).

---

## Concept 5. `@property`: why Python has no getters

In Java you write getters from the start because changing a public field to a
method later breaks every caller. **In Python it does not**, because
`@property` intercepts attribute access at the same syntax.

In [ ]:
class Circle:
    def __init__(self, radius: float) -> None:
        self.radius = radius      # start plain. No getter, no setter.

    @property
    def area(self) -> float:      # a computed, read-only attribute
        return 3.14159 * self.radius ** 2

c = Circle(2)
c.area                            # 12.56...   -- no parentheses
c.area = 5                        # AttributeError: property has no setter

Adding validation later, without changing any call site:

In [ ]:
class Circle:
    def __init__(self, radius: float) -> None:
        self.radius = radius      # this now goes through the setter

    @property
    def radius(self) -> float:
        return self._radius

    @radius.setter
    def radius(self, value: float) -> None:
        if value <= 0:
            raise ValueError(f"radius must be positive, got {value}")
        self._radius = value

Every existing `c.radius` and `c.radius = 5` keeps working, now validated. This
is why **you should not write a getter and setter until you need one.** Start
with a plain attribute; promote it to a property when there is a reason.

Two things to watch:

**Infinite recursion.** Inside the property, use `self._radius`, never
`self.radius` — the latter calls the property again.

**Cheapness.** A property looks like an attribute, so callers assume it is
cheap. A property that issues a database query will be called in a loop by
someone who had no way to know. If it is expensive, make it a method named
`compute_x()`, or cache it:

In [ ]:
from functools import cached_property

class Dataset:
    @cached_property
    def stats(self) -> dict[str, float]:      # computed once, then stored
        return expensive_analysis(self.rows)  # in the instance __dict__

`cached_property` works by writing the result into `self.__dict__`, so step 2 of
the lookup ladder finds it on every subsequent access and the descriptor never
runs again. (Which means it needs a `__dict__` — it does not work with
`__slots__`.)

---

## Concept 8. Encapsulation that actually works

Since `private` does not exist, encapsulation in Python is about **not handing
out mutable internals** — the Module 02 lesson, applied to design.

In [ ]:
class Playlist:
    def __init__(self, tracks: list[str]) -> None:
        self._tracks = list(tracks)          # copy IN

    @property
    def tracks(self) -> tuple[str, ...]:
        return tuple(self._tracks)           # immutable view OUT

    def add(self, track: str) -> None:
        self._tracks.append(track)

Without the copy on the way in, the caller keeps a handle on your internal list.
Without the conversion on the way out, anyone can mutate it. The underscore
documents intent; the copies enforce it.

The alternatives, each with a trade-off:

| Return | Cost | Caller can |
|---|---|---|
| `tuple(self._tracks)` | O(n) copy | read, index, not mutate |
| `list(self._tracks)` | O(n) copy | mutate their own copy |
| `iter(self._tracks)` | O(1) | iterate once; sees later mutations |
| `MappingProxyType(d)` | O(1) | read a dict; a live view, not a snapshot |
| `self._tracks` | O(1) | **everything.** Not encapsulation. |

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: A class body is executable code
- Section 2: The attribute-lookup ladder
- Section 3: Class attributes versus instance attributes
- Section 4: There is no `private`
- Section 5: `@property`: why Python has no getters
- Section 6: `@classmethod` and `@staticmethod`
- Section 7: `__slots__`
- Section 8: Encapsulation that actually works

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

from datetime import datetime
from typing import Any


# --- leak 1 -------------------------------------------------------------------

---

## `ShoppingCart`

_ShoppingCart_

In [ ]:
class ShoppingCart:
    def __init__(self) -> None:
        self._items: list[dict[str, Any]] = []

    def add(self, name: str, price: float, qty: int = 1) -> None:
        self._items.append({"name": name, "price": price, "qty": qty})

    def get_items(self) -> list[dict[str, Any]]:
        return self._items

    @property
    def total(self) -> float:
        return sum(i["price"] * i["qty"] for i in self._items)

---

## `Report`

_Report_

In [ ]:
class Report:
    def __init__(self, rows: list[list[str]]) -> None:
        self._rows = rows

    def row_count(self) -> int:
        return len(self._rows)

---

## `Registry`

_Registry_

In [ ]:
class Registry:
    _handlers: dict[str, Any] = {}          # note: a CLASS attribute

    def register(self, name: str, handler: Any) -> None:
        self._handlers[name] = handler

    def handlers(self) -> dict[str, Any]:
        return self._handlers

---

## `Session`

_Session_

In [ ]:
class Session:
    def __init__(self, user: str) -> None:
        self.user = user
        self.created = datetime.now()
        self._token = "tok_" + user

    def __repr__(self) -> str:
        return f"Session(user={self.user!r}, token={self._token!r})"

    def to_dict(self) -> dict[str, Any]:
        return self.__dict__

---

## `Matrix`

_Matrix_

In [ ]:
class Matrix:
    def __init__(self, rows: int, cols: int) -> None:
        self._data = [[0.0] * cols] * rows     # two bugs in one line

    def get(self, r: int, c: int) -> float:
        return self._data[r][c]

    def set(self, r: int, c: int, value: float) -> None:
        self._data[r][c] = value

---

## `EventLog`

Append-only. Or so the docstring claims.

In [ ]:
class EventLog:
    """Append-only. Or so the docstring claims."""

    def __init__(self) -> None:
        self._events: list[tuple[datetime, str]] = []

    def append(self, message: str) -> None:
        self._events.append((datetime.now(), message))

    def __iter__(self):  # type: ignore[no-untyped-def]
        return iter(self._events)

    def since(self, when: datetime) -> list[tuple[datetime, str]]:
        return [e for e in self._events if e[0] >= when]

    def clear_after_export(self) -> list[tuple[datetime, str]]:
        exported = self._events
        self._events = []
        return exported

---

## `test_cart_does_not_leak`

Prove a caller cannot change the total without going through add().

In [ ]:
def test_cart_does_not_leak() -> None:
    """Prove a caller cannot change the total without going through add()."""
    ...

---

## `test_report_copies_its_input`

_test report copies its input_

In [ ]:
def test_report_copies_its_input() -> None: ...

---

## `test_registry_is_not_shared_across_instances`

_test registry is not shared across instances_

In [ ]:
def test_registry_is_not_shared_across_instances() -> None: ...

---

## `test_session_repr_and_dict_hide_the_token`

_test session repr and dict hide the token_

In [ ]:
def test_session_repr_and_dict_hide_the_token() -> None: ...

---

## `test_matrix_rows_are_independent`

_test matrix rows are independent_

In [ ]:
def test_matrix_rows_are_independent() -> None: ...

---

## `test_event_log_is_really_append_only`

_test event log is really append only_

In [ ]:
def test_event_log_is_really_append_only() -> None: ...

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    # Demonstrate each leak before fixing it.
    c = ShoppingCart()
    c.add("widget", 10.0)
    c.get_items().append({"name": "free stuff", "price": -100.0, "qty": 1})
    print("leak 1: total is now", c.total)

    rows = [["a"]]
    r = Report(rows)
    rows.append(["b"])
    print("leak 2: row_count is now", r.row_count())

    r1, r2 = Registry(), Registry()
    r1.register("x", print)
    print("leak 3: second registry sees", list(r2.handlers()))

    s = Session("ada")
    print("leak 4:", s)
    print("leak 4:", s.to_dict())

    m = Matrix(3, 3)
    m.set(0, 0, 9.0)
    print("leak 5: row 1 is", [m.get(1, c) for c in range(3)])

    log = EventLog()
    log.append("started")
    for entry in log:
        pass
    stolen = log.since(datetime.min)
    stolen.clear()
    print("leak 6: log still has", len(list(log)), "-- but check clear_after_export")

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.